In [47]:
import feedparser
import pandas as pd
import tmdbsimple as tmdb
from sklearn.metrics.pairwise import cosine_similarity
import time
import os
from dotenv import load_dotenv

tmdb.API_KEY = os.getenv("TMDB_API_KEY")
tmdb.REQUESTS_TIMEOUT = 5 

In [48]:
watched_df = pd.read_csv(r"E:\Python\movie-reccomender\ratings.csv")
 
# Parse date and clean up
watched_df["entry_published"] = pd.to_datetime(watched_df["Date"]).dt.strftime("%a, %-d %b %Y %H:%M:%S +0000")
watched_df = watched_df.rename(columns={"Name": "entry_title", "Rating": "entry_rating"})

In [49]:
def search_tmdb(title, year=None):
    """
    Try a movie search first, then fall back to TV.
    Returns (movie_id, tv_id) — one will always be NaN.
    """
    search = tmdb.Search()
 
    # Movie search
    kwargs = {"query": title}
    if year:
        kwargs["year"] = year
    search.movie(**kwargs)
    if search.results:
        return float(search.results[0]["id"]), float("nan")
 
    # TV search (no year filter — TMDB TV search ignores it anyway)
    search.tv(query=title)
    if search.results:
        return float("nan"), float(search.results[0]["id"])
 
    return float("nan"), float("nan")
 
 
movie_ids, tv_ids = [], []
 
for _, row in watched_df.iterrows():
    title = row["entry_title"]
    # Extract year from the Letterboxd "Year" column if present
    year = row.get("Year")
    year = int(year) if pd.notna(year) and str(year).isdigit() else None
 
    m_id, t_id = search_tmdb(title, year)
    movie_ids.append(m_id)
    tv_ids.append(t_id)
 
    time.sleep(0.25)   # stay well within TMDB rate limits (40 req/10 s)
 
watched_df["movie_id"] = movie_ids
watched_df["tv_id"]    = tv_ids

In [50]:
watched_df = watched_df[["entry_title", "entry_published", "entry_rating", "movie_id", "tv_id"]].copy()
watched_df = watched_df.sort_values("entry_published", ascending=False).reset_index(drop=True)
 

watched_df

,entry_title,entry_published,entry_rating,movie_id,tv_id
0,Checkpoint Zoo,2026-04-15 00:00:00,4.0,1176733.0,NaN
1,Project Hail Mary,2026-04-13 00:00:00,4.0,687163.0,NaN
2,The Drama,2026-04-06 00:00:00,4.0,1325734.0,NaN
3,Big Trouble in Little China,2026-02-14 00:00:00,3.0,6978.0,NaN
4,Scream,2026-02-14 00:00:00,4.0,4232.0,NaN
...,...,...,...,...,...
118,The Dark Knight,2024-02-15 00:00:00,5.0,155.0,NaN
119,Nightcrawler,2024-02-07 00:00:00,4.5,242582.0,NaN
120,Risky Business,2024-01-22 00:00:00,4.0,9346.0,NaN
121,The Wolf of Wall Street,2024-01-16 00:00:00,4.0,106646.0,NaN


In [51]:
movie_df = watched_df.dropna(subset = ["movie_id"])
movie_df['tv_id'] = pd.to_numeric(movie_df['tv_id'])
movie_df['movie_id'] = pd.to_numeric(movie_df['movie_id'])



movie_df['genres'] = None  # resets the column to object dtype

for movieId in movie_df['movie_id']:
    movie = tmdb.Movies(int(movieId))
    response = movie.info()
    idx = movie_df[movie_df['movie_id'] == movieId].index[0]
    movie_df.at[idx, 'genres'] = ', '.join([g['name'] for g in movie.genres])
    time.sleep(0.1)

movie_df

,entry_title,entry_published,entry_rating,movie_id,tv_id,genres
0,Checkpoint Zoo,2026-04-15 00:00:00,4.0,1176733.0,NaN,Documentary
1,Project Hail Mary,2026-04-13 00:00:00,4.0,687163.0,NaN,"Science Fiction, Adventure"
2,The Drama,2026-04-06 00:00:00,4.0,1325734.0,NaN,"Romance, Comedy, Drama"
3,Big Trouble in Little China,2026-02-14 00:00:00,3.0,6978.0,NaN,"Action, Adventure, Comedy, Fantasy"
4,Scream,2026-02-14 00:00:00,4.0,4232.0,NaN,"Crime, Horror, Mystery"
...,...,...,...,...,...,...
118,The Dark Knight,2024-02-15 00:00:00,5.0,155.0,NaN,"Action, Crime, Thriller"
119,Nightcrawler,2024-02-07 00:00:00,4.5,242582.0,NaN,"Crime, Drama, Thriller"
120,Risky Business,2024-01-22 00:00:00,4.0,9346.0,NaN,"Romance, Comedy, Drama"
121,The Wolf of Wall Street,2024-01-16 00:00:00,4.0,106646.0,NaN,"Crime, Drama, Comedy"


In [52]:
tv_df = watched_df.dropna(subset = ["tv_id"])
tv_df['movie_id'] = pd.to_numeric(tv_df['movie_id'])
tv_df['tv_id'] = pd.to_numeric(tv_df['tv_id'])

tv_df['genres'] = None  # resets the column to object dtype

for tvId in tv_df['tv_id']:
    tv = tmdb.TV(int(tvId))
    response = tv.info()
    idx = tv_df[tv_df['tv_id'] == tvId].index[0]
    tv_df.at[idx, 'genres'] = ', '.join([g['name'] for g in tv.genres])
    time.sleep(0.1)
tv_df

,entry_title,entry_published,entry_rating,movie_id,tv_id,genres
13,Frieren: Beyond Journey's End,2026-01-19 00:00:00,5.0,NaN,209867.0,"Animation, Action & Adventure, Drama, Sci-Fi &..."
60,Chainsaw Man,2025-05-19 00:00:00,4.0,NaN,114410.0,"Animation, Action & Adventure, Sci-Fi & Fantas..."


In [53]:
movie = tmdb.Movies()
movies = []
page = 1
seen_ids = set(watched_df['movie_id'].dropna().astype(int).tolist())

while len(movies) < 1000:
    response = movie.top_rated(page=page)
    for item in response['results']:
        if item['id'] not in seen_ids:
            movies.append(item)
    page += 1
    time.sleep(.1)

print(response["results"][0]["title"])

len(movies)

popular_df = pd.DataFrame(movies)
popular_df

Avatar


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/zMwhWailP1WY7sb6AoE6b8ugoy.jpg,"[16, 10751, 12, 18, 14]",1007757,Swapped,en,Swapped,"A small woodland creature and a majestic bird,...",439.3294,/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg,2026-05-01,False,False,8.985,718
1,False,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,"[18, 80]",278,The Shawshank Redemption,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,54.5658,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,False,False,8.720,30305
2,False,/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg,"[18, 80]",238,The Godfather,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",43.4483,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,1972-03-14,False,False,8.686,22872
3,False,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,"[18, 80]",240,The Godfather Part II,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,30.4736,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,1974-12-20,False,False,8.571,13869
4,False,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,"[18, 36, 10752]",424,Schindler's List,en,Schindler's List,The true story of how businessman Oskar Schind...,25.9785,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,1993-12-15,False,False,8.568,17407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1008,False,/9n2tJBplPbgR2ca05hS5CKXwP2c.jpg,"[10751, 35, 12, 14, 16]",502356,The Super Mario Bros. Movie,en,The Super Mario Bros. Movie,"While working underground to fix a water main,...",48.7951,/qNBAXBIQlnOThrVvA6mA2B5ggV6.jpg,2023-04-05,False,False,7.594,10633
1009,False,/xXhta1NIKn09IXy0mfp68cabdWS.jpg,"[35, 10749]",466282,To All the Boys I've Loved Before,en,To All the Boys I've Loved Before,Lara Jean's love life goes from imaginary to o...,5.7659,/hKHZhUbIyUAjcSrqJThFGYIR6kI.jpg,2018-08-17,False,False,7.594,8704
1010,False,/v8AmfO3BW4NT4SiNnsocKMzexOR.jpg,"[18, 35, 10749]",61202,Zindagi Na Milegi Dobara,hi,ज़िन्दगी ना मिलेगी दोबारा,Three friends who were inseparable in childhoo...,2.2395,/hKO9O715wYxjkQSEv47giCYcyO8.jpg,2011-07-15,False,False,7.594,405
1011,False,/pZVrDhIPF30JTRSkal2Lfk7NwrI.jpg,"[18, 27, 53]",31417,Eyes Without a Face,fr,Les Yeux sans visage,Dr. Génessier is riddled with guilt after an a...,1.4915,/8y7Z9Gvcq52uOlJlUWyn2epGGRd.jpg,1960-01-11,False,False,7.600,797


In [54]:
#Import TfIdfVectorizer from scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer

#Define a TF-IDF Vectorizer Object. Remove all english stop words such as 'the', 'a'
tfidf = TfidfVectorizer(stop_words='english')

#Replace NaN with an empty string
popular_df['overview'] = popular_df['overview'].fillna('')

#Construct the required TF-IDF matrix by fitting and transforming the data
tfidf_matrix = tfidf.fit_transform(popular_df['overview'])

#Output the shape of tfidf_matrix
tfidf_matrix.shape

(1013, 8372)

In [55]:
# Import linear_kernel
from sklearn.metrics.pairwise import linear_kernel

# Compute the cosine similarity matrix
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [56]:
#Construct a reverse map of indices and movie titles
indices = pd.Series(popular_df.index, index=popular_df['title']).drop_duplicates()
popular_df

,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/zMwhWailP1WY7sb6AoE6b8ugoy.jpg,"[16, 10751, 12, 18, 14]",1007757,Swapped,en,Swapped,"A small woodland creature and a majestic bird,...",439.3294,/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg,2026-05-01,False,False,8.985,718
1,False,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,"[18, 80]",278,The Shawshank Redemption,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,54.5658,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,False,False,8.720,30305
2,False,/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg,"[18, 80]",238,The Godfather,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",43.4483,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,1972-03-14,False,False,8.686,22872
3,False,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,"[18, 80]",240,The Godfather Part II,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,30.4736,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,1974-12-20,False,False,8.571,13869
4,False,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,"[18, 36, 10752]",424,Schindler's List,en,Schindler's List,The true story of how businessman Oskar Schind...,25.9785,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,1993-12-15,False,False,8.568,17407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1008,False,/9n2tJBplPbgR2ca05hS5CKXwP2c.jpg,"[10751, 35, 12, 14, 16]",502356,The Super Mario Bros. Movie,en,The Super Mario Bros. Movie,"While working underground to fix a water main,...",48.7951,/qNBAXBIQlnOThrVvA6mA2B5ggV6.jpg,2023-04-05,False,False,7.594,10633
1009,False,/xXhta1NIKn09IXy0mfp68cabdWS.jpg,"[35, 10749]",466282,To All the Boys I've Loved Before,en,To All the Boys I've Loved Before,Lara Jean's love life goes from imaginary to o...,5.7659,/hKHZhUbIyUAjcSrqJThFGYIR6kI.jpg,2018-08-17,False,False,7.594,8704
1010,False,/v8AmfO3BW4NT4SiNnsocKMzexOR.jpg,"[18, 35, 10749]",61202,Zindagi Na Milegi Dobara,hi,ज़िन्दगी ना मिलेगी दोबारा,Three friends who were inseparable in childhoo...,2.2395,/hKO9O715wYxjkQSEv47giCYcyO8.jpg,2011-07-15,False,False,7.594,405
1011,False,/pZVrDhIPF30JTRSkal2Lfk7NwrI.jpg,"[18, 27, 53]",31417,Eyes Without a Face,fr,Les Yeux sans visage,Dr. Génessier is riddled with guilt after an a...,1.4915,/8y7Z9Gvcq52uOlJlUWyn2epGGRd.jpg,1960-01-11,False,False,7.600,797


In [57]:
# Function that takes in movie title as input and outputs most similar movies
def get_recommendations(title, cosine_sim=cosine_sim):
    # Get the index of the movie that matches the title
    idx = indices[title]

    # Get the pairwsie similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 10 most similar movies
    sim_scores = sim_scores[1:11]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 10 most similar movies
    return popular_df['title'].iloc[movie_indices]

In [62]:
get_recommendations('The Avengers')

948           Kingsman: The Secret Service
923                             Zootopia 2
158    Lock, Stock and Two Smoking Barrels
157                           Paris, Texas
468                       A Beautiful Mind
744                   John Wick: Chapter 4
868                         The Bad Guys 2
933                           Mediterraneo
591            A Woman Under the Influence
488                         Thirteen Lives
Name: title, dtype: str

In [79]:
results = []
for item in popular_df["id"]:
    movie = tmdb.Movies(item)
    credits = movie.credits()
    director = next(
        (member["name"] for member in credits["crew"] if member["job"] == "Director"),
        "Director not found"
    )

    results.append(director)
    time.sleep(.1)
popular_df["director"] = results
popular_df

,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count,director
0,False,/zMwhWailP1WY7sb6AoE6b8ugoy.jpg,"[16, 10751, 12, 18, 14]",1007757,Swapped,en,Swapped,"A small woodland creature and a majestic bird,...",439.3294,/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg,2026-05-01,False,False,8.985,718,Nathan Greno
1,False,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,"[18, 80]",278,The Shawshank Redemption,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,54.5658,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,False,False,8.720,30305,Frank Darabont
2,False,/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg,"[18, 80]",238,The Godfather,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",43.4483,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,1972-03-14,False,False,8.686,22872,Francis Ford Coppola
3,False,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,"[18, 80]",240,The Godfather Part II,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,30.4736,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,1974-12-20,False,False,8.571,13869,Francis Ford Coppola
4,False,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,"[18, 36, 10752]",424,Schindler's List,en,Schindler's List,The true story of how businessman Oskar Schind...,25.9785,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,1993-12-15,False,False,8.568,17407,Steven Spielberg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1008,False,/9n2tJBplPbgR2ca05hS5CKXwP2c.jpg,"[10751, 35, 12, 14, 16]",502356,The Super Mario Bros. Movie,en,The Super Mario Bros. Movie,"While working underground to fix a water main,...",48.7951,/qNBAXBIQlnOThrVvA6mA2B5ggV6.jpg,2023-04-05,False,False,7.594,10633,Michael Jelenic
1009,False,/xXhta1NIKn09IXy0mfp68cabdWS.jpg,"[35, 10749]",466282,To All the Boys I've Loved Before,en,To All the Boys I've Loved Before,Lara Jean's love life goes from imaginary to o...,5.7659,/hKHZhUbIyUAjcSrqJThFGYIR6kI.jpg,2018-08-17,False,False,7.594,8704,Susan Johnson
1010,False,/v8AmfO3BW4NT4SiNnsocKMzexOR.jpg,"[18, 35, 10749]",61202,Zindagi Na Milegi Dobara,hi,ज़िन्दगी ना मिलेगी दोबारा,Three friends who were inseparable in childhoo...,2.2395,/hKO9O715wYxjkQSEv47giCYcyO8.jpg,2011-07-15,False,False,7.594,405,Zoya Akhtar
1011,False,/pZVrDhIPF30JTRSkal2Lfk7NwrI.jpg,"[18, 27, 53]",31417,Eyes Without a Face,fr,Les Yeux sans visage,Dr. Génessier is riddled with guilt after an a...,1.4915,/8y7Z9Gvcq52uOlJlUWyn2epGGRd.jpg,1960-01-11,False,False,7.600,797,Georges Franju
